In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [12]:

import os
import re
import glob
import pandas as pd
import numpy as np

# 1. Automatically find the uploaded Excel file
matches = glob.glob(
    "/kaggle/input/**/TEMI_A_2493924_SM7967.xlsx",
    recursive=True
)

if not matches:
    raise FileNotFoundError(
        "Excel file not found. Please upload "
        "TEMI_A_2493924_SM7967.xlsx to your Kaggle notebook."
    )

file_path = matches[0]
print("File found:", file_path)

# 2. Read the first sheet
raw = pd.read_excel(file_path, sheet_name=0, header=None)

# 3. Extract the six year columns
years = [2017, 2018, 2019, 2020, 2021, 2022]
records = []

for i in range(2, len(raw)):
    ward_text = str(raw.iloc[i, 0])

    match = re.search(r"(\d+)", ward_text)

    if match is None:
        continue

    ward_number = int(match.group(1))

    for j, year in enumerate(years, start=1):
        value = pd.to_numeric(raw.iloc[i, j], errors="coerce")

        if pd.notna(value):
            records.append({
                "ward": ward_number,
                "year": year,
                "dengue_cases_per_10000_population": float(value)
            })

# 4. Create the DataFrame
df = pd.DataFrame(records)

if df.empty:
    raise ValueError("No valid ward data was extracted. Check the Excel file.")

# 5. Calculate relative historical risk ranking per year
df["relative_ward_risk_score_0_100"] = (
    df.groupby("year")[
        "dengue_cases_per_10000_population"
    ].rank(pct=True) * 100
).round(2)

# 6. Assign risk categories
def assign_risk(score):
    if score <= 25:
        return "Low"
    elif score <= 50:
        return "Moderate"
    elif score <= 75:
        return "High"
    else:
        return "Very High"

df["risk_band"] = df[
    "relative_ward_risk_score_0_100"
].apply(assign_risk)

# 7. Add currently unavailable fields as empty values
missing_columns = [
    "rainfall_mm",
    "humidity_mean_percent",
    "temperature_mean_c",
    "mosquito_activity_score",
    "breeding_site_reports"
]

for column in missing_columns:
    df[column] = np.nan

# 8. Add source and data-status information
df["data_frequency"] = "Annual"
df["data_source"] = "Uploaded Kolkata ward-level dengue Excel"
df["risk_score_type"] = "Relative historical ranking"

# 9. Sort and export
df = df.sort_values(
    by=["year", "ward"]
).reset_index(drop=True)

output_path = "/kaggle/working/DengueShield_Demo_Dataset.csv"

df.to_csv(output_path, index=False)

# 10. Verify the result
print("\nCSV CREATED SUCCESSFULLY!")
print("Rows and columns:", df.shape)
print("Unique wards:", df["ward"].nunique())
print("Years:", sorted(df["year"].unique()))
print("\nFirst five rows:")
display(df.head())

print("\nRisk category distribution:")
display(df["risk_band"].value_counts())

print("\nSaved at:", output_path)


File found: /kaggle/input/datasets/shivanshikaranjai/dengue-ward-daily/TEMI_A_2493924_SM7967.xlsx

CSV CREATED SUCCESSFULLY!
Rows and columns: (864, 13)
Unique wards: 143
Years: [np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022)]

First five rows:


,ward,year,dengue_cases_per_10000_population,relative_ward_risk_score_0_100,risk_band,rainfall_mm,humidity_mean_percent,temperature_mean_c,mosquito_activity_score,breeding_site_reports,data_frequency,data_source,risk_score_type
0,1,2017,0.190100,14.58,Low,NaN,NaN,NaN,NaN,NaN,Annual,Uploaded Kolkata ward-level dengue Excel,Relative historical ranking
1,2,2017,0.419129,18.06,Low,NaN,NaN,NaN,NaN,NaN,Annual,Uploaded Kolkata ward-level dengue Excel,Relative historical ranking
2,3,2017,4.740595,61.81,High,NaN,NaN,NaN,NaN,NaN,Annual,Uploaded Kolkata ward-level dengue Excel,Relative historical ranking
3,4,2017,0.000000,6.94,Low,NaN,NaN,NaN,NaN,NaN,Annual,Uploaded Kolkata ward-level dengue Excel,Relative historical ranking
4,5,2017,2.603941,47.22,Moderate,NaN,NaN,NaN,NaN,NaN,Annual,Uploaded Kolkata ward-level dengue Excel,Relative historical ranking



Risk category distribution:


risk_band
Low          216
High         216
Moderate     216
Very High    216
Name: count, dtype: int64


Saved at: /kaggle/working/DengueShield_Demo_Dataset.csv
